# SubsetsC2: KNN vs diffusion as a function of dataset size

**Question.** How much data does KNN retrieval need to match one diffusion counterfactual,
and how much does averaging K neighbours lower that?

**Setup** (`code/c2_knn_size_sweep.py`, model `MLP_I+C+D`, no prob):
- **Test**: the 5 test folds of the 20k diffusion subset (~20k rows), the same for every arm, N, K and seed.
- **Train**: N rows drawn per fold as nested, patient-whole subsamples, 5 seeds.
  - KNN: drawn from all of `c2_data` minus the test-fold patients. The N rows are also the only rows KNN can retrieve from.
  - Diffusion: drawn from the 4 diffusion train folds (max ~16k). One generated CF per query (K=1, `high` level).
- **Score**: AUROC pooled over the 5 test folds, then mean ± SD over seeds.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sys.path.insert(0, os.path.abspath("code"))
from thesis_style import *
apply()

RESULTS  = os.environ.get("THESIS_RESULTS", "/work3/s251710/thesis_results")
C2       = os.path.join(RESULTS, "C2_final_results")
FIG_DIR  = os.path.join("figures", "c2")
DISEASES = ["effusion", "edema", "atelectasis", "cardiomegaly", "consolidation"]
CONFIG   = "I+C+D"
KS_PLOT  = [1, 5, 20]          # KNN K values drawn in the figure
KS_TABLE = [1, 3, 5, 10, 20]
os.makedirs(FIG_DIR, exist_ok=True)

## Load

In [ ]:
def load(disease):
    p = os.path.join(C2, disease, "knn_size_sweep", "sweep_pooled.csv")
    r = pd.read_csv(p)
    r = r[r["config"] == CONFIG].copy()
    r["disease"] = disease
    return r

raw = pd.concat([load(d) for d in DISEASES], ignore_index=True)

# One row per (disease, arm, N, K): mean and SD over seeds. x-axis = the actual mean
# number of training rows (subsamples are patient-whole, so slightly above the target).
agg = (raw.groupby(["disease", "arm", "n_target", "k"])
          .agg(auc=("auc", "mean"), sd=("auc", "std"), n_seeds=("seed", "nunique"),
               n_train=("n_train_mean", "mean"), msp=("msp_auc", "first"))
          .reset_index()
          .sort_values(["disease", "arm", "n_train", "k"]))
MSP = agg.groupby("disease")["msp"].first()
agg.head()

## AUROC by N

In [ ]:
def wide(disease):
    a = agg[agg["disease"] == disease]
    knn = a[a["arm"] == "knn"].pivot_table(index="n_train", columns="k", values="auc")
    knn = knn[[k for k in KS_TABLE if k in knn]].add_prefix("KNN K")
    dif = a[a["arm"] == "diffusion"].set_index("n_train")["auc"].rename("Diffusion K1")
    t = pd.concat([dif, knn], axis=1).sort_index()
    t.index = [f"{n/1000:.1f}k" for n in t.index]
    t.index.name = "N"
    return t

for d in DISEASES:
    print(f"\n{d}   (MSP = {MSP[d]:.4f})")
    display(wide(d).round(4))

## Figure: AUROC vs N

In [ ]:
DIFF_COLOR = "#E08214"
SHOW_DIFF_CURVE = True    # diffusion AUROC vs N; False = only its full-set level as a horizontal line
knn_colors, knn_markers = k_colors("attr", KS_PLOT), k_markers(KS_PLOT)

fig, axes = plt.subplots(1, len(DISEASES), figsize=(FULL, 50 * MM), sharex=True)
for ax, d in zip(axes, DISEASES):
    a = agg[agg["disease"] == d]
    for k in KS_PLOT:
        s = a[(a["arm"] == "knn") & (a["k"] == k)]
        ax.plot(s["n_train"], s["auc"], ls="--", color=knn_colors[k],
                marker=knn_markers[k], ms=3, lw=LW["line"] * 0.8)
    s = a[(a["arm"] == "diffusion") & (a["k"] == 1)].sort_values("n_train")
    if SHOW_DIFF_CURVE:
        ax.plot(s["n_train"], s["auc"], ls="-", color=DIFF_COLOR, marker="o", ms=3.5,
                lw=LW["line"] * 1.4, zorder=5)
    msp_line(ax, MSP[d])

    # Diffusion on its full train side (~16k): the level KNN has to cross.
    target = s["auc"].iloc[-1]
    ax.axhline(target, ls="-", color=DIFF_COLOR, lw=LW["ref"], zorder=4)

    # First N at which a KNN curve reaches it, interpolated in log N between grid points.
    crossings = {}
    for k in KS_PLOT:
        sk = a[(a["arm"] == "knn") & (a["k"] == k)].sort_values("n_train")
        x, y = np.log(sk["n_train"].values), sk["auc"].values
        up = np.where((y[:-1] < target) & (y[1:] >= target))[0]
        if y[0] >= target:
            crossings[k] = sk["n_train"].iloc[0]
        elif len(up):
            i = up[0]
            crossings[k] = np.exp(x[i] + (target - y[i]) / (y[i + 1] - y[i]) * (x[i + 1] - x[i]))
    if crossings:
        k_first = min(crossings, key=crossings.get)
        ax.axvline(crossings[k_first], ls=":", color=knn_colors[k_first], lw=LW["ref"], zorder=4)
        ax.text(crossings[k_first] * 0.92, 0.03, f"K={k_first}\n{crossings[k_first] / 1000:.0f}k",
                transform=ax.get_xaxis_transform(), ha="right", va="bottom",
                fontsize=FS["tiny"], color=knn_colors[k_first])

    ax.set_xscale("log")
    ax.set_xticks([1e3, 4e3, 16e3, 64e3])
    ax.minorticks_off()
    ax.set_xticklabels(["1k", "4k", "16k", "64k"])
    ax.set_title(d)
    ax.set_xlabel("training rows N")
axes[0].set_ylabel("AUROC")

handles = ([Line2D([], [], color=DIFF_COLOR, marker="o", ms=3.5, lw=LW["line"] * 1.4,
                   label="diffusion K=1")] if SHOW_DIFF_CURVE else []) \
          + [Line2D([], [], color=DIFF_COLOR, lw=LW["ref"], label="diffusion, full set (16k)")] \
          + [Line2D([], [], color=knn_colors[k], ls="--", marker=knn_markers[k], ms=3,
                    label=f"KNN K={k}") for k in KS_PLOT] \
          + [msp_handle("MSP")]
fig.legend(handles=handles, loc="lower center", ncol=len(handles),
           bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"subsets_c2_auroc_vs_n.{ext}"))
plt.show()

## Diffusion minus KNN at equal N

Positive = diffusion better. Only N values both arms were run at (≤ ~16k).

In [ ]:
def gap(disease):
    a = agg[agg["disease"] == disease]
    dif = a[a["arm"] == "diffusion"].set_index("n_target")["auc"]
    knn = a[a["arm"] == "knn"].pivot_table(index="n_target", columns="k", values="auc")
    dif = dif.rename({"20k": "16000"})          # the full diffusion train side
    common = [n for n in dif.index if n in knn.index]
    g = knn.loc[common, [k for k in KS_TABLE if k in knn]].rsub(dif.loc[common], axis=0)
    g = g.add_prefix("K").loc[sorted(common, key=int)]
    g.index = [f"{int(n)/1000:.0f}k" for n in g.index]
    g.index.name = "N"
    return g

for d in DISEASES:
    print(f"\n{d}")
    display(gap(d).round(4))

## Figure: diffusion minus KNN vs N

Gap = AUROC(diffusion K=1) − AUROC(KNN at K), computed per seed and then mean ± SD over seeds.
Above zero = diffusion better. The last panel averages the gap over the five diseases.

In [ ]:
GAP_KS = [1, 5, 20]
gap_colors, gap_markers = k_colors("attr", GAP_KS), k_markers(GAP_KS)

r = raw.assign(n=raw["n_target"].replace({"20k": "16000"}))   # full diffusion train side = 16k
dif = r[r["arm"] == "diffusion"].pivot_table(index=["disease", "n"], columns="seed", values="auc")

gaps = []   # one row per (disease, N, K, seed)
for k in GAP_KS:
    knn = r[(r["arm"] == "knn") & (r["k"] == k)].pivot_table(
        index=["disease", "n"], columns="seed", values="auc")
    g = (dif - knn.reindex(dif.index)).stack().rename("gap").reset_index()
    g["k"] = k
    gaps.append(g)
gaps = pd.concat(gaps, ignore_index=True).dropna()
gaps["n"] = gaps["n"].astype(int)

panels = DISEASES + ["mean over diseases"]
fig, axes = plt.subplots(1, len(panels), figsize=(FULL, 50 * MM), sharex=True, sharey=True)
for ax, d in zip(axes, panels):
    g = gaps if d == panels[-1] else gaps[gaps["disease"] == d]
    if d == panels[-1]:
        # average over diseases within each seed first, so the SD is still across seeds
        g = g.groupby(["n", "k", "seed"])["gap"].mean().reset_index()
    s = g.groupby(["k", "n"])["gap"].agg(["mean", "std"]).reset_index()
    ax.axhline(0, color=INK, lw=LW["axis"], zorder=1)
    for k in GAP_KS:
        sk = s[s["k"] == k].sort_values("n")
        ax.fill_between(sk["n"], sk["mean"] - sk["std"], sk["mean"] + sk["std"],
                        color=gap_colors[k], alpha=0.15, lw=0)
        ax.plot(sk["n"], sk["mean"], color=gap_colors[k], marker=gap_markers[k], ms=3)
    ax.set_xscale("log")
    ax.set_xticks([1e3, 2e3, 4e3, 8e3, 16e3])
    ax.set_xticklabels(["1k", "2k", "4k", "8k", "16k"])
    ax.minorticks_off()
    ax.set_title(d)
    ax.set_xlabel("training rows N")
axes[0].set_ylabel("AUROC gap\n(diffusion − KNN)")

handles = [Line2D([], [], color=gap_colors[k], marker=gap_markers[k], ms=3,
                  label=f"vs KNN K={k}") for k in GAP_KS]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"subsets_c2_gap_vs_n.{ext}"))
plt.show()

## How much data KNN needs to match full diffusion

Smallest N at which KNN (mean over seeds) reaches diffusion K=1 trained on the full ~16k.
`>all` means it never got there within the pool.

In [ ]:
rows = []
for d in DISEASES:
    a = agg[agg["disease"] == d]
    target = a[(a["arm"] == "diffusion") & (a["n_target"] == "20k")]["auc"].iloc[0]
    row = {"disease": d, "diffusion@16k": round(target, 4)}
    for k in KS_TABLE:
        s = a[(a["arm"] == "knn") & (a["k"] == k)].sort_values("n_train")
        hit = s[s["auc"] >= target]
        row[f"K{k}"] = f"{hit['n_train'].iloc[0]/1000:.0f}k" if len(hit) else ">all"
    rows.append(row)
pd.DataFrame(rows).set_index("disease")

In [ ]:
nstar = pd.DataFrame(rows).set_index("disease")
k_cols = [f"K{k}" for k in KS_TABLE]

def tex_n(v):
    return r"$>$all" if v == ">all" else v

lines = [
    r"\begin{table}[htbp]", r"\centering", r"\small",
    r"\caption{Training rows KNN needs to match one diffusion counterfactual. "
    r"Smallest $N$ at which the KNN attribute model (\texttt{I+C+D}, mean over 5 seeds) "
    r"reaches the AUROC of the diffusion model ($K=1$) trained on the full 16k train side. "
    r"AUROC is computed out-of-fold over the 20k diffusion subset (5-fold CV: each image is "
    r"tested once, by the model whose fold held it out), with identical folds for every cell.}",
    r"\label{tab:subsets_knn_nstar}",
    r"\begin{tabular}{l c " + "c" * len(k_cols) + "}",
    r"\toprule",
    r" & & \multicolumn{" + str(len(k_cols)) + r"}{c}{KNN $N^*$ at $K =$} \\",
    r"\cmidrule(lr){3-" + str(2 + len(k_cols)) + "}",
    r"Disease & Diffusion AUROC & " + " & ".join(str(k) for k in KS_TABLE) + r" \\",
    r"\midrule",
]
for d, r_ in nstar.iterrows():
    lines.append(f"{d.capitalize()} & {r_['diffusion@16k']:.3f} & "
                 + " & ".join(tex_n(r_[c]) for c in k_cols) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
print("\n".join(lines))